# Pure water/steam cooling and condensation inside tubes (v0.6.2)

Cooling and condensing **pure H2O** (no carrier gas) flowing inside bare tubes: a
steam heater / desuperheater-condenser-subcooler. Unlike v0.6.1's wet-gas
condensation model (dew point, humidity ratio `W`, Chilton-Colburn/Lewis mass
transfer), this is a direct vapor-liquid phase-equilibrium problem parameterized by
vapor quality `x`, solved with `IAPWS97WaterSteamProvider` on the inside.

Four practical cases (A-D), then two cases (E-F) dedicated to the v0.6.2 patch that
replaced Shah (1979) with Shah (2009) as the production in-tube condensation
correlation:

- **A.** Saturated vapor inlet -> partial condensation.
- **B.** Wet steam inlet (`x_in < 1`) -> further condensation to a lower `x_out`.
- **C.** Superheated steam inlet -> desuperheating + condensation.
- **D.** Superheated steam inlet, large surface -> complete condensation +
  condensate subcooling.
- **E.** Low mass flux (`G` below Shah 1979's own documented floor of
  10.8 kg/(m2*s)) -- Shah (2009) selects a gravity-augmented regime instead of
  extrapolating the forced-convective-only 1979 correlation.
- **F.** High mass flux (inside Shah 1979's own validated range) -- Shah (2009)
  selects the pure forced-convective Regime I, the same physics 1979 covered.

Each case reports: `phase_in`/`phase_out`, `T_in`/`T_out`, `T_sat`, `h_in`/`h_out`,
`quality_in`/`quality_out`, `Q_desuperheat`/`Q_condensation`/`Q_subcooling`/`Q_total`,
area fractions, zone heat-transfer coefficients, warnings, and the two-phase
pressure-drop support status. Cases E/F additionally show the selected Shah (2009)
regime and its dimensionless groups directly from the correlation module.


In [1]:
import sys
from pathlib import Path

repository_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'core').is_dir())
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from core.geometry.bundle import TubeBundle
from core.geometry.tube import BareTube, TubeOrientation
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.properties.gas_mixture import GasMixturePropertyProvider, GasMixtureSpec
from core.properties.water import IAPWS97WaterSteamProvider, WATER_CRITICAL_PRESSURE_PA, water_steam_state
from core.heat_transfer.condensation_inside_shah2009 import (
    CORRELATION_NAME as SHAH2009_CORRELATION_NAME,
    shah2009_condensation_alpha_local,
)

P = 101_325.0  # Pa


def make_hx(n_rows, n_tubes_per_row, length_total, flow_arrangement='crossflow',
            orientation=TubeOrientation.HORIZONTAL, D_i=0.020, D_o=0.025,
            pitch_transverse=0.040, pitch_longitudinal=0.040, layout='staggered'):
    tube = BareTube(
        D_i=D_i, D_o=D_o, length_total=length_total, length_effective=length_total,
        wall_k=50.0, tube_orientation=orientation,
    )
    bundle = TubeBundle(
        tube=tube, n_rows=n_rows, n_tubes_per_row=n_tubes_per_row,
        pitch_transverse=pitch_transverse, pitch_longitudinal=pitch_longitudinal,
        layout=layout, n_passes_tube=1, flow_arrangement=flow_arrangement,
    )
    return BareTubeHeatExchanger(bundle)


def dry_air_provider():
    return GasMixturePropertyProvider(GasMixtureSpec(components={'N2': 0.79, 'O2': 0.21}, basis='mole'))


def summarize(label, hx, inside, result):
    """Print the case's key inputs/outputs, including the mass flux G and
    tube inner diameter D_i that feed the Shah (2009) condensation
    correlation (production default since the v0.6.2 low-mass-flux patch;
    Shah 1979 remains available as a legacy/reference implementation but
    is no longer called in production -- see
    core.heat_transfer.condensation_inside_shah2009 module docstring).
    Shown explicitly here since a past bug (fixed earlier in v0.6.2) left
    the top-level alfa_i/UA_actual/thermal_state fields showing the
    sensible-only dry baseline's single-phase HTC instead of the real
    multi-zone condensation physics.
    """
    pc = result.inside_phase_change
    D_i = hx.bundle.tube.D_i
    G = inside.m_dot / hx.bundle.internal_flow_area_per_pass
    print(f"=== {label} ===")
    print(f"D_i={D_i*1000:.2f} mm   G={G:.3f} kg/(m2*s)   m_dot={inside.m_dot:.4f} kg/s   orientation={hx.bundle.tube.tube_orientation.value}")
    print(f"phase_in={pc.phase_in.value:<18s} phase_out={pc.phase_out.value}")
    print(f"T_in={pc.T_in:8.3f} K   T_out={pc.T_out:8.3f} K   T_sat={pc.T_sat:8.3f} K")
    print(f"h_in={pc.h_in:12.1f} J/kg   h_out={pc.h_out:12.1f} J/kg")
    print(f"quality_in={pc.quality_in}   quality_out={pc.quality_out}")
    print(f"Q_desuperheat={pc.Q_desuperheat:10.1f} W   Q_condensation={pc.Q_condensation:10.1f} W   Q_subcooling={pc.Q_subcooling:10.1f} W   Q_total={pc.Q_total:10.1f} W")
    print(f"f_desuperheat={pc.f_desuperheat:.4f}   f_condensation={pc.f_condensation:.4f}   f_subcooling={pc.f_subcooling:.4f}")
    print(f"zone alpha [W/(m2*K)] (real per-zone HTC): desuperheat={pc.zone_alpha_desuperheat}, condensation={pc.zone_alpha_condensation}, subcooling={pc.zone_alpha_subcooling}")
    print(f"result.inside_alfa_mean (EQUIVALENT multi-zone HTC, NOT the condensation coefficient -- see docstring) = {result.inside_alfa_mean:.3f} W/(m2*K)")
    print(f"two_phase_pressure_drop_supported={pc.two_phase_pressure_drop_supported}")
    print('warnings:')
    for w in pc.warnings:
        print(f"  [{w.severity}] {w.code}: {w.message}")
    print()
    return pc


def shah2009_regime_chain(*, p, D_i, G, orientation, x):
    """Independent, solver-free reconstruction of the Shah (2009) chain at
    one local quality x -- for display only (does not feed Rating/Simulation).
    """
    sat_liquid = water_steam_state(p=p, x=0.0)
    sat_vapor = water_steam_state(p=p, x=1.0)
    r = shah2009_condensation_alpha_local(
        x, p=p, p_critical=WATER_CRITICAL_PRESSURE_PA, G=G, D_i=D_i, orientation=orientation,
        mu_L=sat_liquid.mu, mu_G=sat_vapor.mu, k_L=sat_liquid.k, cp_L=sat_liquid.cp,
        rho_L=sat_liquid.rho, rho_G=sat_vapor.rho,
    )
    print(f"correlation={SHAH2009_CORRELATION_NAME}   x={x:.2f}   regime={r.regime}")
    print(f"Jg={r.Jg:.4f}   Z={r.Z:.4f}   Re_LT={r.Re_LT:.1f}   Re_LS={r.Re_LS:.1f}   Re_GT={r.Re_GT:.1f}   Pr_L={r.Pr_L:.4f}   p_r={r.p_r:.5f}")
    print(f"h_I={r.h_I:.2f} W/(m2*K)   h_Nu={r.h_Nu:.2f} W/(m2*K)   local_alpha={r.alpha:.2f} W/(m2*K)")
    for w in r.warnings:
        print(f"  [{w.severity}] {w.code}")
    return r


## Case A -- saturated vapor inlet -> partial condensation

INPUT: saturated steam (`x_in=1.0`) at 1 atm, `m_dot=0.15 kg/s`, cooled by dry air at 290 K.

In [2]:
hx_a = make_hx(n_rows=4, n_tubes_per_row=6, length_total=6.0)
inside_a = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.15, x_in=1.0, p=P)
outside_a = HXSideInput(provider=dry_air_provider(), m_dot=2.0, T_in=290.0, p=P)

result_a = hx_a.simulate(inside_a, outside_a)
pc_a = summarize('A. Saturated vapor -> partial condensation', hx_a, inside_a, result_a)
assert pc_a.active and 0.0 < pc_a.quality_out < 1.0


=== A. Saturated vapor -> partial condensation ===
D_i=20.00 mm   G=19.894 kg/(m2*s)   m_dot=0.1500 kg/s   orientation=horizontal
phase_in=saturated_vapor    phase_out=two_phase
T_in= 373.124 K   T_out= 373.124 K   T_sat= 373.124 K
h_in=   2675531.5 J/kg   h_out=   2500346.7 J/kg
quality_in=1.0   quality_out=0.922365783713758
Q_desuperheat=       0.0 W   Q_condensation=   26277.7 W   Q_subcooling=       0.0 W   Q_total=   26277.7 W
f_desuperheat=0.0000   f_condensation=1.0000   f_subcooling=0.0000
zone alpha [W/(m2*K)] (real per-zone HTC): desuperheat=None, condensation=21238.030435276913, subcooling=None
result.inside_alfa_mean (EQUIVALENT multi-zone HTC, NOT the condensation coefficient -- see docstring) = 21238.030 W/(m2*K)
two_phase_pressure_drop_supported=False
warnings:
  [warning] SHAH_2009_CONDENSATION_OUT_OF_RANGE: Z = 0.00471708 - is outside the Shah (2009) correlation's documented applicability range [0.005, 20] -; this result is an extrapolation.
  [info] TWO_PHASE_PRESSURE

## Case B -- wet steam inlet -> further condensation

INPUT: wet steam at `x_in=0.7` (already partially condensed upstream) at 1 atm, `m_dot=0.15 kg/s`,
cooled by dry air at 290 K.

In [3]:
hx_b = make_hx(n_rows=4, n_tubes_per_row=6, length_total=6.0)
inside_b = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.15, x_in=0.7, p=P)
outside_b = HXSideInput(provider=dry_air_provider(), m_dot=2.0, T_in=290.0, p=P)

result_b = hx_b.simulate(inside_b, outside_b)
pc_b = summarize('B. Wet steam inlet -> lower x_out', hx_b, inside_b, result_b)
assert pc_b.quality_out < pc_b.quality_in


=== B. Wet steam inlet -> lower x_out ===
D_i=20.00 mm   G=19.894 kg/(m2*s)   m_dot=0.1500 kg/s   orientation=horizontal
phase_in=two_phase          phase_out=two_phase
T_in= 373.124 K   T_out= 373.124 K   T_sat= 373.124 K
h_in=   1998569.2 J/kg   h_out=   1823592.4 J/kg
quality_in=0.7   quality_out=0.6224579508416355
Q_desuperheat=       0.0 W   Q_condensation=   26246.5 W   Q_subcooling=       0.0 W   Q_total=   26246.5 W
f_desuperheat=0.0000   f_condensation=1.0000   f_subcooling=0.0000
zone alpha [W/(m2*K)] (real per-zone HTC): desuperheat=None, condensation=12336.41704209174, subcooling=None
result.inside_alfa_mean (EQUIVALENT multi-zone HTC, NOT the condensation coefficient -- see docstring) = 12336.417 W/(m2*K)
two_phase_pressure_drop_supported=False
warnings:
  [info] TWO_PHASE_PRESSURE_DROP_NOT_SUPPORTED: inside: two-phase condensation-zone pressure drop is not modelled in v0.6.2; single-phase tube-side dp components do not represent the full condensing-zone dp.
  [info] INSID

## Case C -- superheated steam inlet -> desuperheating + condensation

INPUT: superheated steam at 450 K, 1 atm, `m_dot=0.15 kg/s`, cooled by dry air at 290 K.

In [4]:
hx_c = make_hx(n_rows=4, n_tubes_per_row=6, length_total=6.0)
inside_c = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.15, T_in=450.0, p=P)
outside_c = HXSideInput(provider=dry_air_provider(), m_dot=2.0, T_in=290.0, p=P)

result_c = hx_c.simulate(inside_c, outside_c)
pc_c = summarize('C. Superheated -> desuperheating + condensation', hx_c, inside_c, result_c)
assert pc_c.Q_desuperheat > 0.0 and pc_c.Q_condensation > 0.0


=== C. Superheated -> desuperheating + condensation ===
D_i=20.00 mm   G=19.894 kg/(m2*s)   m_dot=0.1500 kg/s   orientation=horizontal
phase_in=superheated_vapor  phase_out=two_phase
T_in= 450.000 K   T_out= 373.124 K   T_sat= 373.124 K
h_in=   2829672.0 J/kg   h_out=   2641861.3 J/kg
quality_in=None   quality_out=0.9850788442417979
Q_desuperheat=   23121.1 W   Q_condensation=    5050.5 W   Q_subcooling=       0.0 W   Q_total=   28171.6 W
f_desuperheat=0.8072   f_condensation=0.1928   f_subcooling=0.0000
zone alpha [W/(m2*K)] (real per-zone HTC): desuperheat=115.05265585022195, condensation=30561.66460016311, subcooling=None
result.inside_alfa_mean (EQUIVALENT multi-zone HTC, NOT the condensation coefficient -- see docstring) = 151.395 W/(m2*K)
two_phase_pressure_drop_supported=False
warnings:
  [warning] SHAH_2009_CONDENSATION_OUT_OF_RANGE: Z = 0.00390966 - is outside the Shah (2009) correlation's documented applicability range [0.005, 20] -; this result is an extrapolation.
  [info] 

## Case D -- superheated steam, large surface -> complete condensation + subcooling

INPUT: superheated steam at 450 K, 1 atm, `m_dot=0.4 kg/s`, a larger exchanger and colder outside
air (280 K) so the exchanger fully condenses the steam and subcools the condensate.

In [5]:
hx_d = make_hx(n_rows=16, n_tubes_per_row=16, length_total=20.0)
inside_d = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.4, T_in=450.0, p=P)
outside_d = HXSideInput(provider=dry_air_provider(), m_dot=50.0, T_in=280.0, p=P)

result_d = hx_d.simulate(inside_d, outside_d)
pc_d = summarize('D. Superheated -> complete condensation -> subcooled condensate', hx_d, inside_d, result_d)
assert pc_d.Q_desuperheat > 0.0 and pc_d.Q_condensation > 0.0 and pc_d.Q_subcooling > 0.0
assert pc_d.quality_out is None  # subcooled liquid


=== D. Superheated -> complete condensation -> subcooled condensate ===
D_i=20.00 mm   G=4.974 kg/(m2*s)   m_dot=0.4000 kg/s   orientation=horizontal
phase_in=superheated_vapor  phase_out=subcooled_liquid
T_in= 450.000 K   T_out= 291.618 K   T_sat= 373.124 K
h_in=   2829672.0 J/kg   h_out=     77602.5 J/kg
quality_in=None   quality_out=None
Q_desuperheat=   61656.2 W   Q_condensation=  902616.3 W   Q_subcooling=  136555.3 W   Q_total= 1100827.8 W
f_desuperheat=0.0645   f_condensation=0.4311   f_subcooling=0.5044
zone alpha [W/(m2*K)] (real per-zone HTC): desuperheat=37.24736856746274, condensation=10682.440882072584, subcooling=123.92890715242872
result.inside_alfa_mean (EQUIVALENT multi-zone HTC, NOT the condensation coefficient -- see docstring) = 246.179 W/(m2*K)
two_phase_pressure_drop_supported=False
warnings:
  [warning] SHAH_2009_CONDENSATION_OUT_OF_RANGE: Jg = 0.0220199 - is outside the Shah (2009) correlation's documented applicability range [0.06, 20] -; this result is an ext

## Case E -- low mass flux (Shah 2009 gravity-augmented regime)

INPUT: saturated steam at 14 bara, many parallel narrow tubes (`D_i=15 mm`, matching a
real steam-heater bundle) at low per-tube mass flux (`G` well below Shah 1979's own
documented floor of 10.8 kg/(m2*s)). This is the exact scenario that motivated the
v0.6.2 low-mass-flux patch: the legacy Shah (1979) correlation has no gravity-film
branch and, extrapolated here, predicts a condensation HTC of only a few hundred
W/(m2*K) -- physically far too low for condensing steam. Shah (2009) instead selects
Regime II (forced convection + gravity-film) or Regime III (pure gravity/Nusselt,
vertical/inclined tubes only) and predicts an order of magnitude higher, physically
plausible coefficient -- **without any calibration factor.**

In [6]:
hx_e = make_hx(
    n_rows=10, n_tubes_per_row=40, length_total=1.5, D_i=0.015, D_o=0.018,
    pitch_transverse=0.025, pitch_longitudinal=0.025, layout='inline',
)
inside_e = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=900.81 / 3600.0, x_in=1.0, p=14.0e5)
outside_e = HXSideInput(provider=dry_air_provider(), m_dot=28_785.0 / 3600.0, T_in=298.15, p=P)

result_e = hx_e.simulate(inside_e, outside_e)
pc_e = summarize('E. Low mass flux -> gravity-augmented condensation', hx_e, inside_e, result_e)
G_e = inside_e.m_dot / hx_e.bundle.internal_flow_area_per_pass
print(f"G_e = {G_e:.3f} kg/(m2*s) -- below Shah 1979's documented floor of 10.8 kg/(m2*s)")
r_e = shah2009_regime_chain(p=14.0e5, D_i=hx_e.bundle.tube.D_i, G=G_e, orientation=hx_e.bundle.tube.tube_orientation, x=0.5)
assert r_e.regime != 'I'  # gravity contribution must be active at this G
assert G_e < 10.8


=== E. Low mass flux -> gravity-augmented condensation ===
D_i=15.00 mm   G=3.540 kg/(m2*s)   m_dot=0.2502 kg/s   orientation=horizontal
phase_in=saturated_vapor    phase_out=subcooled_liquid
T_in= 468.197 K   T_out= 447.976 K   T_sat= 468.197 K
h_in=   2788893.0 J/kg   h_out=    740656.3 J/kg
quality_in=1.0   quality_out=None
Q_desuperheat=       0.0 W   Q_condensation=  490130.9 W   Q_subcooling=   22389.1 W   Q_total=  512520.0 W
f_desuperheat=0.0000   f_condensation=0.9162   f_subcooling=0.0838
zone alpha [W/(m2*K)] (real per-zone HTC): desuperheat=None, condensation=13510.492561214087, subcooling=161.80806327340485
result.inside_alfa_mean (EQUIVALENT multi-zone HTC, NOT the condensation coefficient -- see docstring) = 2746.278 W/(m2*K)
two_phase_pressure_drop_supported=False
warnings:
  [warning] SHAH_2009_CONDENSATION_OUT_OF_RANGE: Pr_L = 0.931376 - is outside the Shah (2009) correlation's documented applicability range [1, 18] -; this result is an extrapolation.
  [warning] SHAH

## Case F -- high mass flux (Shah 2009 Regime I, matches Shah 1979's own range)

INPUT: same pressure/inlet quality as Case E, but a coarser bundle (few wide tubes)
driving a much higher per-tube mass flux, inside Shah 1979's own documented range
(10.8-210.6 kg/(m2*s)). Here Shah (2009) selects the pure forced-convective Regime I
(`h_TP=h_I`, no gravity-film addition) -- the same physics the legacy correlation
covered well.

In [7]:
hx_f = make_hx(
    n_rows=2, n_tubes_per_row=2, length_total=3.0, D_i=0.020, D_o=0.025,
    pitch_transverse=0.040, pitch_longitudinal=0.040, layout='staggered',
)
inside_f = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.19, x_in=1.0, p=14.0e5)
outside_f = HXSideInput(provider=dry_air_provider(), m_dot=5.0, T_in=298.15, p=P)

result_f = hx_f.simulate(inside_f, outside_f)
pc_f = summarize('F. High mass flux -> forced-convective Regime I', hx_f, inside_f, result_f)
G_f = inside_f.m_dot / hx_f.bundle.internal_flow_area_per_pass
print(f"G_f = {G_f:.3f} kg/(m2*s) -- inside Shah 1979's documented range [10.8, 210.6] kg/(m2*s)")
r_f = shah2009_regime_chain(p=14.0e5, D_i=hx_f.bundle.tube.D_i, G=G_f, orientation=hx_f.bundle.tube.tube_orientation, x=0.5)
assert r_f.regime == 'I'
assert 10.8 <= G_f <= 210.6


=== F. High mass flux -> forced-convective Regime I ===
D_i=20.00 mm   G=151.197 kg/(m2*s)   m_dot=0.1900 kg/s   orientation=horizontal
phase_in=saturated_vapor    phase_out=two_phase
T_in= 468.197 K   T_out= 468.197 K   T_sat= 468.197 K
h_in=   2788893.0 J/kg   h_out=   2670929.9 J/kg
quality_in=1.0   quality_out=0.9397766655310988
Q_desuperheat=       0.0 W   Q_condensation=   22413.0 W   Q_subcooling=       0.0 W   Q_total=   22413.0 W
f_desuperheat=0.0000   f_condensation=1.0000   f_subcooling=0.0000
zone alpha [W/(m2*K)] (real per-zone HTC): desuperheat=None, condensation=19761.16082344173, subcooling=None
result.inside_alfa_mean (EQUIVALENT multi-zone HTC, NOT the condensation coefficient -- see docstring) = 19761.161 W/(m2*K)
two_phase_pressure_drop_supported=False
warnings:
  [warning] SHAH_2009_CONDENSATION_OUT_OF_RANGE: Pr_L = 0.931376 - is outside the Shah (2009) correlation's documented applicability range [1, 18] -; this result is an extrapolation.
  [info] TWO_PHASE_PRESS

## Rating: closing a known heat balance (Case D)

Reuses the same steam-side inlet/outlet resolution and multi-zone physics as
Simulation -- ``Rating`` is only the "given both temperature programs, find the
required area / overdesign factor" orchestration on top.

In [8]:
from core.models.heat_balance import BalanceSideSpec

# Independent, comfortably-achievable closed balance (counterflow, so the
# maximum-achievable-effectiveness ceiling is not a binding constraint here).
hx_rating = make_hx(n_rows=16, n_tubes_per_row=16, length_total=20.0, flow_arrangement='counterflow')

rating_d = hx_rating.rate(
    BalanceSideSpec(provider=IAPWS97WaterSteamProvider(), p=P, m_dot=0.4, T_in=450.0, T_out=300.0),
    BalanceSideSpec(provider=dry_air_provider(), p=P, m_dot=50.0, T_in=280.0, T_out=300.0),
)
rpc_d = rating_d.inside_phase_change
print(f"overdesign_factor={rating_d.overdesign_factor:.4f}")
print(f"A_required (outside-area basis) [m2]={rating_d.A_required:.4f}")
print(f"Q_required [W]={rpc_d.Q_total:.1f}")
print(f"phase_out={rpc_d.phase_out.value}")


overdesign_factor=0.3868
A_required (outside-area basis) [m2]=289.9641
Q_required [W]=1086802.8
phase_out=subcooled_liquid
